In [3]:

!pip install ultralytics deep-sort-realtime opencv-python-headless -q
 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 90.6 MB/s eta 0:00:00:00:0100:01


In [4]:

import cv2
import numpy as np
import math
import time
import os
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from collections import defaultdict
 
print("✅ All imports successful")
 
 

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ All imports successful


In [14]:
SPEED_LIMIT_KMH = 15       
INPUT_VIDEO     = "/kaggle/input/datasets/saadhassan118/videooftraffic/0414.mp4"   
OUTPUT_VIDEO    = "/kaggle/working/output_speed.mp4"    
 

VEHICLE_CLASSES   = [2, 3, 5, 7]   
AVG_CAR_LENGTH_M  = 4.5            
CONFIDENCE        = 0.4            
FRAME_SKIP        = 1              
MAX_TRACK_AGE     = 30            
HISTORY_LEN       = 8             
 
print("✅ Configuration set")
print(f"   Speed limit : {SPEED_LIMIT_KMH} km/h")
print(f"   Input video : {INPUT_VIDEO}")
print(f"   Output video: {OUTPUT_VIDEO}")

✅ Configuration set
   Speed limit : 15 km/h
   Input video : /kaggle/input/datasets/saadhassan118/videooftraffic/0414.mp4
   Output video: /kaggle/working/output_speed.mp4


In [6]:

class PixelScaleEstimator:
   
    def __init__(self, avg_car_length_m=4.5, sample_limit=30):
        self.avg_car_length_m = avg_car_length_m
        self.sample_limit     = sample_limit
        self.samples          = []
        self.locked_scale     = None   
 
    def update(self, bbox_width_px, bbox_height_px, class_id):
       
        if self.locked_scale is not None:
            return                         
 
        if class_id != 2:                
            return
 
       
        longest_px = max(bbox_width_px, bbox_height_px)
        if longest_px < 30:            
            return
 
        ppm = longest_px / self.avg_car_length_m
        self.samples.append(ppm)
 
        if len(self.samples) >= self.sample_limit:
            # Take the median to reject outliers
            self.locked_scale = float(np.median(self.samples))
            print(f"✅ Pixel scale locked: {self.locked_scale:.2f} px/m  "
                  f"(from {len(self.samples)} samples)")
 
    @property
    def pixels_per_meter(self):
        if self.locked_scale is not None:
            return self.locked_scale
        if len(self.samples) > 0:
            return float(np.median(self.samples))   # live estimate before lock
        return None                                  # not enough data yet
 
    @property
    def is_ready(self):
        return len(self.samples) >= 3               # need at least 3 samples

In [7]:
 
class VehicleSpeedTracker:
    """
    Tracks position history for each vehicle ID and computes smoothed speed.
    """
    def __init__(self, history_len=8, fps=30):
        self.history_len = history_len
        self.fps         = fps
        # {track_id: [(cx, cy, frame_number), ...]}
        self.positions   = defaultdict(list)
        self.speeds      = defaultdict(float)   # last computed speed
 
    def update(self, track_id, cx, cy, frame_number, pixels_per_meter):
        hist = self.positions[track_id]
        hist.append((cx, cy, frame_number))
 
        # Keep only last N positions
        if len(hist) > self.history_len:
            hist.pop(0)
 
        if len(hist) < 2 or pixels_per_meter is None:
            return 0.0
 
        # Compute speed from oldest to newest point in history
        x0, y0, f0 = hist[0]
        x1, y1, f1 = hist[-1]
 
        pixel_dist  = math.hypot(x1 - x0, y1 - y0)
        meter_dist  = pixel_dist / pixels_per_meter
        elapsed_sec = (f1 - f0) / self.fps
 
        if elapsed_sec <= 0:
            return 0.0
 
        speed_kmh = (meter_dist / elapsed_sec) * 3.6
        speed_kmh = min(speed_kmh, 250.0)          # cap at 250 km/h (outlier guard)
 
        # Exponential moving average for smoothness
        prev = self.speeds[track_id]
        self.speeds[track_id] = 0.6 * prev + 0.4 * speed_kmh
 
        return self.speeds[track_id]
 
    def get_speed(self, track_id):
        return self.speeds.get(track_id, 0.0)
 

In [8]:

def draw_box(frame, x1, y1, x2, y2, speed_kmh, track_id, speed_limit, label):
    """Draw bounding box and speed label. Red if over limit, green if safe."""
    over_limit = speed_kmh > speed_limit
    color      = (0, 0, 220) if over_limit else (0, 200, 0)   # BGR: red / green
    thickness  = 2
 
    # Main bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
 
    # Label background
    speed_text = f"{label} #{track_id}  {speed_kmh:.0f} km/h"
    if over_limit:
        speed_text += "  ⚠ OVER LIMIT"
 
    (tw, th), _ = cv2.getTextSize(speed_text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    label_y = max(y1 - 6, th + 4)
    cv2.rectangle(frame,
                  (x1, label_y - th - 4),
                  (x1 + tw + 6, label_y + 2),
                  color, -1)
 
    # Text (white on colored background)
    cv2.putText(frame, speed_text,
                (x1 + 3, label_y - 1),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (255, 255, 255), 1, cv2.LINE_AA)
 
 
def draw_hud(frame, frame_num, total_frames, fps,
             speed_limit, ppm, vehicle_count, violation_count):
    """Draw heads-up display in top-left corner."""
    h, w = frame.shape[:2]
    lines = [
        f"Speed limit : {speed_limit} km/h",
        f"Px / meter  : {ppm:.1f}" if ppm else "Px / meter  : calibrating...",
        f"Vehicles    : {vehicle_count}",
        f"Violations  : {violation_count}",
        f"Frame       : {frame_num}/{total_frames}",
        f"FPS (proc)  : {fps:.1f}",
    ]
 
    pad, lh = 10, 22
    box_h = len(lines) * lh + pad * 2
    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (260, 8 + box_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
 
    for i, line in enumerate(lines):
        y = 8 + pad + (i + 1) * lh - 4
        cv2.putText(frame, line, (16, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52,
                    (200, 255, 200), 1, cv2.LINE_AA)
 

In [9]:

def process_video(input_path, output_path, speed_limit):
 
    # Load models
    print("⏳ Loading YOLOv8 model...")
    model   = YOLO("yolov8n.pt")          # downloads automatically ~6 MB
    tracker = DeepSort(max_age=MAX_TRACK_AGE)
    print("✅ Models loaded")
 
    # Open video
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {input_path}")
 
    fps          = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
 
    print(f"📹 Video: {width}×{height} @ {fps:.1f} fps  |  {total_frames} frames")
 
    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
 
    # Helpers
    scale_estimator = PixelScaleEstimator(avg_car_length_m=AVG_CAR_LENGTH_M)
    speed_tracker   = VehicleSpeedTracker(history_len=HISTORY_LEN, fps=fps)
 
    CLASS_NAMES = {2: "Car", 3: "Bike", 5: "Bus", 7: "Truck"}
 
    frame_num        = 0
    violation_count  = 0
    active_vehicles  = set()
    proc_start       = time.time()
 
    while True:
        ret, frame = cap.read()
        if not ret:
            break
 
        frame_num += 1
 
        # ── Detection ──────────────────────────────────────
        results    = model(frame, classes=VEHICLE_CLASSES,
                           conf=CONFIDENCE, verbose=False)[0]
        detections = []
 
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf            = float(box.conf[0])
            cls             = int(box.cls[0])
            w_px            = x2 - x1
            h_px            = y2 - y1
 
            # Update pixel scale estimator
            scale_estimator.update(w_px, h_px, cls)
 
            detections.append(([x1, y1, w_px, h_px], conf, cls))
 
        # ── Tracking ───────────────────────────────────────
        tracks = tracker.update_tracks(detections, frame=frame)
 
        active_vehicles.clear()
        current_violations = 0
 
        for track in tracks:
            if not track.is_confirmed():
                continue
 
            tid  = track.track_id
            ltrb = track.to_ltrb()
            x1, y1, x2, y2 = map(int, ltrb)
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            cls    = track.det_class if hasattr(track, "det_class") else 2
 
            # Update speed
            ppm   = scale_estimator.pixels_per_meter
            speed = speed_tracker.update(tid, cx, cy, frame_num, ppm)
 
            active_vehicles.add(tid)
            if speed > speed_limit:
                current_violations += 1
 
            label = CLASS_NAMES.get(cls, "Vehicle")
            draw_box(frame, x1, y1, x2, y2,
                     speed, tid, speed_limit, label)
 
        violation_count = max(violation_count, current_violations)
 
        # ── HUD ────────────────────────────────────────────
        elapsed  = time.time() - proc_start
        proc_fps = frame_num / elapsed if elapsed > 0 else 0
        draw_hud(frame, frame_num, total_frames, proc_fps,
                 speed_limit,
                 scale_estimator.pixels_per_meter,
                 len(active_vehicles),
                 current_violations)
 
        out.write(frame)
 
        # Progress every 100 frames
        if frame_num % 100 == 0:
            pct = frame_num / total_frames * 100
            ppm_str = f"{scale_estimator.pixels_per_meter:.1f}" \
                      if scale_estimator.pixels_per_meter else "..."
            print(f"  [{pct:5.1f}%]  frame {frame_num}/{total_frames}"
                  f"  |  px/m: {ppm_str}"
                  f"  |  vehicles: {len(active_vehicles)}"
                  f"  |  violations: {current_violations}")
 
    cap.release()
    out.release()
 
    total_time = time.time() - proc_start
    print(f"\n✅ Done! Processed {frame_num} frames in {total_time:.1f}s")
    print(f"📁 Output saved → {output_path}")
    return output_path
 

In [15]:
output = process_video(
    input_path  = INPUT_VIDEO,
    output_path = OUTPUT_VIDEO,
    speed_limit = SPEED_LIMIT_KMH,
)

⏳ Loading YOLOv8 model...
✅ Models loaded
📹 Video: 1920×1080 @ 30.0 fps  |  717 frames
✅ Pixel scale locked: 56.67 px/m  (from 30 samples)
  [ 13.9%]  frame 100/717  |  px/m: 56.7  |  vehicles: 0  |  violations: 0
  [ 27.9%]  frame 200/717  |  px/m: 56.7  |  vehicles: 8  |  violations: 0
  [ 41.8%]  frame 300/717  |  px/m: 56.7  |  vehicles: 9  |  violations: 4
  [ 55.8%]  frame 400/717  |  px/m: 56.7  |  vehicles: 12  |  violations: 5
  [ 69.7%]  frame 500/717  |  px/m: 56.7  |  vehicles: 14  |  violations: 2
  [ 83.7%]  frame 600/717  |  px/m: 56.7  |  vehicles: 14  |  violations: 3
  [ 97.6%]  frame 700/717  |  px/m: 56.7  |  vehicles: 13  |  violations: 4

✅ Done! Processed 717 frames in 56.5s
📁 Output saved → /kaggle/working/output_speed.mp4
